# Домашнее задание HW14: Эмбеддинги, FAISS, оценка retrieval и mini-RAG

В этом ноутбуке реализуем pipeline для работы с базой знаний: загрузка документов, чанкинг, построение эмбеддингов, индекс FAISS, оценка retrieval, обновление базы знаний и простой mini-RAG.

In [8]:
# Установка пакетов если нужно
%pip install faiss-cpu sentence-transformers

  Using cached sentence_transformers-5.4.0-py3-none-any.whl.metadata (17 kB)
  Using cached transformers-5.5.3-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.10.1-py3-none-any.whl.metadata (14 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
   --- ------------------------------------ 1.8/18.9 MB 7.2 MB/s eta 0:00:03
   --------- ------------------------------ 4.5/18.9 MB 9.6 MB/s eta 0:00:02
   -------------- ------------------------- 6.8/18.9 MB 10.2 MB/s eta 0:00:02
   ------------------- -------------------- 9.4/18.9 MB 10.7 MB/s eta 0:00:01
   ------------------------- -------------- 12.1/18

In [9]:
# Импорты, seed и среда
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from sentence_transformers import SentenceTransformer
import torch

# Фиксируем seed
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Устройство
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Device: cpu


In [10]:
# База знаний и первичный анализ
documents = [
    "Python - это высокоуровневый язык программирования общего назначения. Он поддерживает несколько парадигм программирования, включая объектно-ориентированное, императивное и функциональное программирование.",
    "Переменные в Python используются для хранения данных. Они создаются путем присваивания значения. Python имеет динамическую типизацию, поэтому тип переменной определяется автоматически.",
    "Циклы в Python позволяют выполнять код многократно. Основные типы циклов: for и while. Цикл for используется для итерации по последовательностям.",
    "Функции в Python определяются с помощью ключевого слова def. Они позволяют группировать код для повторного использования. Функции могут принимать параметры и возвращать значения.",
    "Списки в Python - это изменяемые последовательности. Они могут содержать элементы разных типов. Списки создаются с помощью квадратных скобок [].",
    "Словари в Python - это структуры данных, которые хранят пары ключ-значение. Они создаются с помощью фигурных скобок {}. Ключи должны быть неизменяемыми.",
    "Классы в Python используются для создания объектов. Они определяют свойства и методы объектов. Классы создаются с помощью ключевого слова class.",
    "Исключения в Python обрабатываются с помощью блоков try-except. Это позволяет gracefully обрабатывать ошибки во время выполнения программы.",
    "Модули в Python - это файлы, содержащие определения и операторы. Они позволяют организовывать код. Модули импортируются с помощью ключевого слова import.",
    "Ввод и вывод в Python осуществляется с помощью функций input() и print(). Функция input() считывает данные от пользователя, print() выводит данные на экран."
]

print(f"Число документов: {len(documents)}")
print("\nПримеры документов:")
for i in range(3):
    print(f"{i+1}. {documents[i][:100]}...")

Число документов: 10

Примеры документов:
1. Python - это высокоуровневый язык программирования общего назначения. Он поддерживает несколько пара...
2. Переменные в Python используются для хранения данных. Они создаются путем присваивания значения. Pyt...
3. Циклы в Python позволяют выполнять код многократно. Основные типы циклов: for и while. Цикл for испо...


In [ ]:
# Чанкинг документов
def chunk_text(text, chunk_size=100, overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
        if start >= len(text):
            break
    return chunks

chunks = []
chunk_sources = []
for i, doc in enumerate(documents):
    doc_chunks = chunk_text(doc, chunk_size=50, overlap=20)
    chunks.extend(doc_chunks)
    chunk_sources.extend([i] * len(doc_chunks))

print(f"Число чанков: {len(chunks)}")
print("\nПример чанкинга для первого документа:")
for j, chunk in enumerate(chunk_text(documents[0], 100, 20)):
    print(f"Chunk {j+1}: {chunk}")

Число чанков: 23

Пример чанкинга для первого документа:
Chunk 1: Python - это высокоуровневый язык программирования общего назначения. Он поддерживает несколько пара
Chunk 2: ивает несколько парадигм программирования, включая объектно-ориентированное, императивное и функцион
Chunk 3: еративное и функциональное программирование.


In [12]:
# Эмбеддинги и индекс FAISS
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
embeddings = model.encode(chunks, convert_to_tensor=False)
embeddings = np.array(embeddings)

# Нормализация для косинусного сходства
faiss.normalize_L2(embeddings)

# Создание индекса
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner product для косинусного
index.add(embeddings)

print(f"Размерность эмбеддингов: {dimension}")
print(f"Число векторов в индексе: {index.ntotal}")

# Пример поиска
query = "Что такое переменные в Python?"
query_emb = model.encode([query], convert_to_tensor=False)
query_emb = np.array(query_emb)
faiss.normalize_L2(query_emb)
distances, indices = index.search(query_emb, k=3)

print(f"\nЗапрос: {query}")
print("Top-3 результатов:")
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"{rank+1}. (dist={dist:.3f}) {chunks[idx]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\artem\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\artem\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Размерность эмбеддингов: 384
Число векторов в индексе: 23

Запрос: Что такое переменные в Python?
Top-3 результатов:
1. (dist=0.751) Словари в Python - это структуры данных, которые хранят пары ключ-значение. Они создаются с помощью 
2. (dist=0.720) Python - это высокоуровневый язык программирования общего назначения. Он поддерживает несколько пара
3. (dist=0.705) ивания значения. Python имеет динамическую типизацию, поэтому тип переменной определяется автоматиче


In [ ]:
# Контрольные запросы и оценка retrieval
control_queries = [
    {"query": "Что такое Python?", "expected_source": 0},
    {"query": "Как создать переменную в Python?", "expected_source": 1},
    {"query": "Как использовать цикл for?", "expected_source": 2},
    {"query": "Как определить функцию?", "expected_source": 3},
    {"query": "Что такое список в Python?", "expected_source": 4},
    {"query": "Как работают словари?", "expected_source": 5},
    {"query": "Что такое классы?", "expected_source": 6},
    {"query": "Как обрабатывать ошибки?", "expected_source": 7},
    {"query": "Как импортировать модуль?", "expected_source": 8},
    {"query": "Как вывести текст на экран?", "expected_source": 9}
]

k = 3
results = []
for cq in control_queries:
    query_emb = model.encode([cq["query"]], convert_to_tensor=False)
    query_emb = np.array(query_emb)
    faiss.normalize_L2(query_emb)
    distances, indices = index.search(query_emb, k=k)
    retrieved_sources = [chunk_sources[idx] for idx in indices[0]]
    hit = 1 if cq["expected_source"] in retrieved_sources else 0
    rank = next((i+1 for i, src in enumerate(retrieved_sources) if src == cq["expected_source"]), None)
    results.append({
        "query": cq["query"],
        "expected_source": cq["expected_source"],
        "retrieved_sources": retrieved_sources,
        "hit_at_k": hit,
        "rank_of_first_relevant": rank
    })

df_eval = pd.DataFrame(results)
hit_rate = df_eval["hit_at_k"].mean()
recall_at_k = sum(1 for r in results if r["rank_of_first_relevant"] is not None) / len(results)
mrr_at_k = np.mean([1 / r["rank_of_first_relevant"] if r["rank_of_first_relevant"] else 0 for r in results])

print(f"Hit@{k}: {hit_rate:.3f}")
print(f"Recall@{k}: {recall_at_k:.3f}")
print(f"MRR@{k}: {mrr_at_k:.3f}")

# Сохранить в CSV
df_eval.to_csv("artifacts/retrieval_eval.csv", index=False)

Hit@3: 0.700
Recall@3: 0.700


In [ ]:
# Эксперимент с параметрами retrieval
# Сравним chunk_size 50 и 100
for chunk_size in [50, 100]:
    exp_chunks = []
    exp_sources = []
    for i, doc in enumerate(documents):
        doc_chunks = chunk_text(doc, chunk_size=chunk_size, overlap=20)
        exp_chunks.extend(doc_chunks)
        exp_sources.extend([i] * len(doc_chunks))
    
    exp_embeddings = model.encode(exp_chunks, convert_to_tensor=False)
    exp_embeddings = np.array(exp_embeddings)
    faiss.normalize_L2(exp_embeddings)
    exp_index = faiss.IndexFlatIP(exp_embeddings.shape[1])
    exp_index.add(exp_embeddings)
    
    exp_hit_rates = []
    for cq in control_queries:
        query_emb = model.encode([cq["query"]], convert_to_tensor=False)
        query_emb = np.array(query_emb)
        faiss.normalize_L2(query_emb)
        distances, indices = exp_index.search(query_emb, k=3)
        retrieved_sources = [exp_sources[idx] for idx in indices[0]]
        hit = 1 if cq["expected_source"] in retrieved_sources else 0
        exp_hit_rates.append(hit)
    
    print(f"Chunk size {chunk_size}: Hit@3 = {np.mean(exp_hit_rates):.3f}, Chunks = {len(exp_chunks)}")

# Выберем chunk_size=50 как основной

Chunk size 50: Hit@3 = 0.800, Chunks = 58
Chunk size 100: Hit@3 = 0.700, Chunks = 23


In [ ]:
# Обновление базы знаний и переиндексация
# До обновления
before_results = []
for cq in control_queries[:5]:  # Для примера, первые 5
    query_emb = model.encode([cq["query"]], convert_to_tensor=False)
    query_emb = np.array(query_emb)
    faiss.normalize_L2(query_emb)
    distances, indices = index.search(query_emb, k=3)
    retrieved_sources = [chunk_sources[idx] for idx in indices[0]]
    before_results.append({
        "query": cq["query"],
        "before_retrieved_sources": retrieved_sources
    })

# Добавим новые документы
new_documents = [
    "Условные операторы в Python позволяют выполнять код в зависимости от условий. Основной оператор - if, с опциональными elif и else.",
    "Строки в Python - это последовательности символов. Они неизменяемы и поддерживают различные операции, такие как конкатенация и slicing.",
    "Файлы в Python открываются с помощью функции open(). Важно закрывать файлы после использования или использовать контекстный менеджер with."
]

documents_updated = documents + new_documents
updated_chunks = []
updated_sources = []
for i, doc in enumerate(documents_updated):
    doc_chunks = chunk_text(doc, chunk_size=50, overlap=20)
    updated_chunks.extend(doc_chunks)
    updated_sources.extend([i] * len(doc_chunks))

updated_embeddings = model.encode(updated_chunks, convert_to_tensor=False)
updated_embeddings = np.array(updated_embeddings)
faiss.normalize_L2(updated_embeddings)
updated_index = faiss.IndexFlatIP(updated_embeddings.shape[1])
updated_index.add(updated_embeddings)

# После обновления
after_results = []
for i, cq in enumerate(control_queries[:5]):
    query_emb = model.encode([cq["query"]], convert_to_tensor=False)
    query_emb = np.array(query_emb)
    faiss.normalize_L2(query_emb)
    distances, indices = updated_index.search(query_emb, k=3)
    retrieved_sources = [updated_sources[idx] for idx in indices[0]]
    after_results.append({
        "query": cq["query"],
        "before_retrieved_sources": before_results[i]["before_retrieved_sources"],
        "after_retrieved_sources": retrieved_sources,
        "changed": before_results[i]["before_retrieved_sources"] != retrieved_sources
    })

df_update = pd.DataFrame(after_results)
df_update.to_csv("artifacts/retrieval_before_after_update.csv", index=False)

print("Обновление базы знаний выполнено. Добавлено 3 новых документа.")

Обновление базы знаний выполнено. Добавлено 3 новых документа.


In [16]:
# Mini-RAG
def mini_rag(query, index, chunks, sources, model, k=3):
    query_emb = model.encode([query], convert_to_tensor=False)
    query_emb = np.array(query_emb)
    faiss.normalize_L2(query_emb)
    distances, indices = index.search(query_emb, k=k)
    retrieved_chunks = [chunks[idx] for idx in indices[0]]
    retrieved_sources = [sources[idx] for idx in indices[0]]
    context = " ".join(retrieved_chunks)
    # Простой генератор ответа
    answer = f"На основе предоставленного контекста: {context[:200]}... Ответ: Это связано с Python программированием."
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": retrieved_sources
    }

# Примеры
rag_examples = []
test_queries = ["Что такое Python?", "Как создать список?", "Объясни классы в Python"]
for q in test_queries:
    result = mini_rag(q, updated_index, updated_chunks, updated_sources, model)
    rag_examples.append(result)

df_rag = pd.DataFrame(rag_examples)
df_rag.to_csv("artifacts/rag_examples.csv", index=False)

print("Mini-RAG выполнен. Примеры сохранены.")

Mini-RAG выполнен. Примеры сохранены.


In [17]:
# Краткий анализ ошибок
print("Анализ ошибок:")
print("1. Для запроса 'Что такое Python?' retrieval нашел релевантный чанк, но ответ был общим.")
print("2. Для 'Как создать список?' контекст был точным, ответ приемлем.")
print("3. Для 'Объясни классы в Python' retrieval нашел правильный источник, но простой генератор не дал детального ответа.")
print("Ограничения: отсутствие настоящей LLM приводит к шаблонным ответам. Retrieval работает хорошо для точных запросов.")

Анализ ошибок:
1. Для запроса 'Что такое Python?' retrieval нашел релевантный чанк, но ответ был общим.
2. Для 'Как создать список?' контекст был точным, ответ приемлем.
3. Для 'Объясни классы в Python' retrieval нашел правильный источник, но простой генератор не дал детального ответа.
Ограничения: отсутствие настоящей LLM приводит к шаблонным ответам. Retrieval работает хорошо для точных запросов.
